# Práctica SQL 02 — Caso 01: E-commerce Andino (preguntas)

10 preguntas básicas y 20 intermedias.

**Plataforma:** Databricks SQL · Delta Lake · Unity Catalog

## Descripción

Una empresa peruana de comercio electrónico vende productos de varias categorías mediante web, app y tiendas físicas. La gerencia necesita analizar clientes, pedidos, vendedores, ventas, descuentos, tiempos de entrega y devoluciones.

### Regla de negocio para los ejercicios

Salvo que la pregunta indique lo contrario, considera como **venta válida** todo pedido cuyo estado sea diferente de `cancelado`. El importe neto de una línea se calcula así:

```sql
cantidad * precio_unitario * (1 - descuento_pct / 100)
```

## Tablas

| Tabla | Columnas y tipos | Descripción |
|---|---|---|
| `clientes` | `cliente_id STRING`; `nombre_cliente STRING`; `ciudad STRING`; `region STRING`; `segmento STRING`; `fecha_registro DATE`; `email STRING` | Información demográfica y fecha de registro. |
| `vendedores` | `vendedor_id STRING`; `nombre_vendedor STRING`; `region STRING`; `fecha_ingreso DATE`; `meta_mensual DECIMAL(12,2)` | Región y meta mensual de cada vendedor. |
| `productos` | `producto_id STRING`; `nombre_producto STRING`; `categoria STRING`; `marca STRING`; `precio_lista DECIMAL(12,2)`; `costo_unitario DECIMAL(12,2)`; `activo BOOLEAN` | Catálogo, precios y costos. |
| `pedidos` | `pedido_id STRING`; `cliente_id STRING`; `vendedor_id STRING`; `fecha_pedido DATE`; `canal STRING`; `metodo_pago STRING`; `estado STRING`; `ciudad_envio STRING`; `fecha_entrega DATE` *(admite NULL)* | Cabecera del pedido, canal, estado y entrega. |
| `detalle_pedidos` | `pedido_id STRING`; `linea_id INT`; `producto_id STRING`; `cantidad INT`; `precio_unitario DECIMAL(12,2)`; `descuento_pct DECIMAL(5,2)` | Productos, cantidades, precios y descuentos. |
| `devoluciones` | `devolucion_id STRING`; `pedido_id STRING`; `producto_id STRING`; `fecha_devolucion DATE`; `cantidad_devuelta INT`; `motivo STRING`; `monto_reembolso DECIMAL(12,2)` | Productos devueltos y montos reembolsados. |

## Relaciones

- `clientes.cliente_id` → `pedidos.cliente_id` (1 a N)
- `vendedores.vendedor_id` → `pedidos.vendedor_id` (1 a N)
- `pedidos.pedido_id` → `detalle_pedidos.pedido_id` (1 a N)
- `productos.producto_id` → `detalle_pedidos.producto_id` (1 a N)
- `pedidos.pedido_id` → `devoluciones.pedido_id` (1 a N)
- `productos.producto_id` → `devoluciones.producto_id` (1 a N)

## Consideraciones

- Los archivos TXT usan formato CSV con encabezado y codificación UTF-8.
- Las fechas usan el formato `YYYY-MM-DD`.
- Los campos vacíos representan `NULL`, principalmente `fecha_entrega` (pedidos aún no entregados).
- Los pedidos cancelados conservan sus líneas para permitir analizar intentos de compra, pero **no** se consideran ventas válidas.

---
## PARTE 0 — Preparación del entorno (DDL y carga)

> Si ya creaste y cargaste las tablas de este caso, salta directamente a la **PARTE 1**.

### 0.1 Esquemas y volumen

| Objeto | Nombre | Para qué sirve |
|---|---|---|
| Esquema | `workspace.sql_practice_raw` | Contiene el volumen con los archivos originales. |
| Volumen | `workspace.sql_practice_raw.datasets` | Almacena los `.txt` de ambos casos, cada uno en su carpeta. |
| Esquema | `workspace.sql_practice_ecommerce` | Contiene las 6 tablas de este caso. |

En Databricks Free Edition el catálogo por defecto es `workspace`. Si el tuyo se llama `main` u otro nombre, reemplaza `workspace` en todo el notebook, incluidas las rutas `/Volumes/workspace/...`.

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.sql_practice_raw;
CREATE VOLUME IF NOT EXISTS workspace.sql_practice_raw.datasets;

CREATE SCHEMA IF NOT EXISTS workspace.sql_practice_ecommerce;

### 0.2 Subir los archivos al volumen

1. En **Catalog Explorer** abre el volumen `workspace` → `sql_practice_raw` → **Volumes** → `datasets`.
2. Crea la carpeta **`caso_01_ecommerce_andino`** (si aún no existe).
3. Dentro de esa carpeta, sube los 6 archivos de `datos_txt/`: `clientes.txt`, `vendedores.txt`, `productos.txt`, `pedidos.txt`, `detalle_pedidos.txt`, `devoluciones.txt`.
4. Ejecuta la siguiente celda: deben aparecer los 6 archivos.

In [0]:
%sql
LIST '/Volumes/workspace/sql_practice_raw/datasets/caso_01_ecommerce_andino/';

path,name,size,modification_time
/Volumes/workspace/sql_practice_raw/datasets/caso_01_ecommerce_andino/clientes.txt,clientes.txt,9738,1792723907000
/Volumes/workspace/sql_practice_raw/datasets/caso_01_ecommerce_andino/detalle_pedidos.txt,detalle_pedidos.txt,31341,1792723907000
/Volumes/workspace/sql_practice_raw/datasets/caso_01_ecommerce_andino/devoluciones.txt,devoluciones.txt,3774,1792723907000
/Volumes/workspace/sql_practice_raw/datasets/caso_01_ecommerce_andino/pedidos.txt,pedidos.txt,24461,1792723907000
/Volumes/workspace/sql_practice_raw/datasets/caso_01_ecommerce_andino/productos.txt,productos.txt,2680,1792723907000
/Volumes/workspace/sql_practice_raw/datasets/caso_01_ecommerce_andino/vendedores.txt,vendedores.txt,617,1792723907000


### 0.3 Eliminar tablas previas

Permite volver a ejecutar la preparación desde cero. Las tablas se eliminan en **orden inverso a sus dependencias**: primero las hijas y luego las padres.

In [0]:
%sql
USE CATALOG workspace;
USE SCHEMA sql_practice_ecommerce;

DROP TABLE IF EXISTS devoluciones;
DROP TABLE IF EXISTS detalle_pedidos;
DROP TABLE IF EXISTS pedidos;
DROP TABLE IF EXISTS productos;
DROP TABLE IF EXISTS vendedores;
DROP TABLE IF EXISTS clientes;

### 0.4 Crear las tablas (DDL)

- Todas las columnas son `NOT NULL`, salvo las indicadas como *admite NULL*. Delta Lake **sí** valida esta restricción al insertar.
- `PRIMARY KEY` y `FOREIGN KEY` en Unity Catalog son restricciones **informativas**: documentan el modelo, pero **no se validan** al insertar. Por eso en el paso 0.6 verificamos la integridad con consultas.
- Las tablas se crean de **padres a hijas**: una clave foránea solo puede apuntar a una tabla que ya tenga su clave primaria.

#### `clientes`

In [0]:
%sql
CREATE TABLE clientes (
  cliente_id      STRING  NOT NULL COMMENT 'Identificador del cliente',
  nombre_cliente  STRING  NOT NULL COMMENT 'Nombre completo',
  ciudad          STRING  NOT NULL COMMENT 'Ciudad de residencia',
  region          STRING  NOT NULL COMMENT 'Región del país',
  segmento        STRING  NOT NULL COMMENT 'Segmento del cliente',
  fecha_registro  DATE    NOT NULL COMMENT 'Fecha de registro',
  email           STRING  NOT NULL COMMENT 'Correo electrónico (ficticio)',
  CONSTRAINT pk_clientes PRIMARY KEY (cliente_id)
)
COMMENT 'Clientes de la tienda online.';

#### `vendedores`

In [0]:
%sql
CREATE TABLE vendedores (
  vendedor_id      STRING         NOT NULL COMMENT 'Identificador del vendedor',
  nombre_vendedor  STRING         NOT NULL COMMENT 'Nombre completo',
  region           STRING         NOT NULL COMMENT 'Región asignada',
  fecha_ingreso    DATE           NOT NULL COMMENT 'Fecha de ingreso a la empresa',
  meta_mensual     DECIMAL(12,2)  NOT NULL COMMENT 'Meta de ventas mensual en soles',
  CONSTRAINT pk_vendedores PRIMARY KEY (vendedor_id)
)
COMMENT 'Vendedores y su meta mensual.';

#### `productos`

In [0]:
%sql
CREATE TABLE productos (
  producto_id      STRING         NOT NULL COMMENT 'Identificador del producto',
  nombre_producto  STRING         NOT NULL COMMENT 'Nombre del producto',
  categoria        STRING         NOT NULL COMMENT 'Categoría del producto',
  marca            STRING         NOT NULL COMMENT 'Marca',
  precio_lista     DECIMAL(12,2)  NOT NULL COMMENT 'Precio de lista en soles',
  costo_unitario   DECIMAL(12,2)  NOT NULL COMMENT 'Costo unitario en soles',
  activo           BOOLEAN        NOT NULL COMMENT 'Indica si el producto está activo',
  CONSTRAINT pk_productos PRIMARY KEY (producto_id)
)
COMMENT 'Catálogo de productos.';

#### `pedidos`

In [0]:
%sql
CREATE TABLE pedidos (
  pedido_id      STRING  NOT NULL COMMENT 'Identificador del pedido',
  cliente_id     STRING  NOT NULL COMMENT 'Cliente que realiza el pedido',
  vendedor_id    STRING  NOT NULL COMMENT 'Vendedor responsable',
  fecha_pedido   DATE    NOT NULL COMMENT 'Fecha del pedido',
  canal          STRING  NOT NULL COMMENT 'Canal de venta',
  metodo_pago    STRING  NOT NULL COMMENT 'Método de pago',
  estado         STRING  NOT NULL COMMENT 'Estado del pedido',
  ciudad_envio   STRING  NOT NULL COMMENT 'Ciudad de envío',
  fecha_entrega  DATE             COMMENT 'Fecha de entrega (NULL si aún no se entrega)',
  CONSTRAINT pk_pedidos PRIMARY KEY (pedido_id),
  CONSTRAINT fk_pedidos_clientes FOREIGN KEY (cliente_id) REFERENCES clientes (cliente_id),
  CONSTRAINT fk_pedidos_vendedores FOREIGN KEY (vendedor_id) REFERENCES vendedores (vendedor_id)
)
COMMENT 'Cabecera de pedidos.';

#### `detalle_pedidos`

In [0]:
%sql
CREATE TABLE detalle_pedidos (
  pedido_id        STRING         NOT NULL COMMENT 'Pedido al que pertenece la línea',
  linea_id         INT            NOT NULL COMMENT 'Número de línea dentro del pedido',
  producto_id      STRING         NOT NULL COMMENT 'Producto vendido',
  cantidad         INT            NOT NULL COMMENT 'Unidades vendidas',
  precio_unitario  DECIMAL(12,2)  NOT NULL COMMENT 'Precio unitario aplicado en soles',
  descuento_pct    DECIMAL(5,2)   NOT NULL COMMENT 'Descuento en porcentaje (0 a 100)',
  CONSTRAINT pk_detalle_pedidos PRIMARY KEY (pedido_id, linea_id),
  CONSTRAINT fk_detalle_pedidos_pedidos FOREIGN KEY (pedido_id) REFERENCES pedidos (pedido_id),
  CONSTRAINT fk_detalle_pedidos_productos FOREIGN KEY (producto_id) REFERENCES productos (producto_id)
)
COMMENT 'Líneas de cada pedido.';

#### `devoluciones`

In [0]:
%sql
CREATE TABLE devoluciones (
  devolucion_id      STRING         NOT NULL COMMENT 'Identificador de la devolución',
  pedido_id          STRING         NOT NULL COMMENT 'Pedido de origen',
  producto_id        STRING         NOT NULL COMMENT 'Producto devuelto',
  fecha_devolucion   DATE           NOT NULL COMMENT 'Fecha de la devolución',
  cantidad_devuelta  INT            NOT NULL COMMENT 'Unidades devueltas',
  motivo             STRING         NOT NULL COMMENT 'Motivo de la devolución',
  monto_reembolso    DECIMAL(12,2)  NOT NULL COMMENT 'Monto reembolsado en soles',
  CONSTRAINT pk_devoluciones PRIMARY KEY (devolucion_id),
  CONSTRAINT fk_devoluciones_pedidos FOREIGN KEY (pedido_id) REFERENCES pedidos (pedido_id),
  CONSTRAINT fk_devoluciones_productos FOREIGN KEY (producto_id) REFERENCES productos (producto_id)
)
COMMENT 'Productos devueltos y montos reembolsados.';

### 0.5 Cargar los datos

Usamos `read_files` para leer cada `.txt` del volumen:

- `format => 'csv'` y `header => true`: texto delimitado por comas con encabezado.
- `schema => '...'`: define el tipo de cada columna en lugar de dejar que Databricks lo infiera. Los campos vacíos se cargan como `NULL`.
- `mode => 'FAILFAST'`: si alguna fila no se puede convertir al tipo indicado, la carga falla en lugar de dejar datos incorrectos.

#### `clientes`

In [0]:
%sql
INSERT INTO clientes
SELECT
  cliente_id,
  nombre_cliente,
  ciudad,
  region,
  segmento,
  fecha_registro,
  email
FROM read_files(
  '/Volumes/workspace/sql_practice_raw/datasets/caso_01_ecommerce_andino/clientes.txt',
  format => 'csv',
  header => true,
  mode => 'FAILFAST',
  schema => 'cliente_id STRING, nombre_cliente STRING, ciudad STRING, region STRING, segmento STRING, fecha_registro DATE, email STRING'
);

SELECT * FROM clientes LIMIT 5;

cliente_id,nombre_cliente,ciudad,region,segmento,fecha_registro,email
C0001,Renzo Reyes,Cusco,Sur,Persona,2024-03-06,renzo.reyes1@example.com
C0002,Natalia Aguilar,Piura,Norte,Persona,2024-12-04,natalia.aguilar2@example.com
C0003,Hugo Núñez,Cajamarca,Norte,Empresa,2024-10-12,hugo.nunez3@example.com
C0004,Carla Mendoza,Cusco,Sur,Persona,2025-08-26,carla.mendoza4@example.com
C0005,Carlos Silva,Puno,Sur,Persona,2024-01-19,carlos.silva5@example.com


#### `vendedores`

In [0]:
%sql
INSERT INTO vendedores
SELECT
  vendedor_id,
  nombre_vendedor,
  region,
  fecha_ingreso,
  meta_mensual
FROM read_files(
  '/Volumes/workspace/sql_practice_raw/datasets/caso_01_ecommerce_andino/vendedores.txt',
  format => 'csv',
  header => true,
  mode => 'FAILFAST',
  schema => 'vendedor_id STRING, nombre_vendedor STRING, region STRING, fecha_ingreso DATE, meta_mensual DECIMAL(12,2)'
);

SELECT * FROM vendedores LIMIT 5;

vendedor_id,nombre_vendedor,region,fecha_ingreso,meta_mensual
V001,José Cárdenas,Costa,2022-10-29,25000.00
V002,José Salazar,Norte,2024-07-29,15000.00
V003,Fernando Torres,Centro,2023-06-09,30000.00
V004,Camila Herrera,Sur,2023-08-16,18000.00
V005,Paola Silva,Costa,2025-01-21,30000.00


#### `productos`

In [0]:
%sql
INSERT INTO productos
SELECT
  producto_id,
  nombre_producto,
  categoria,
  marca,
  precio_lista,
  costo_unitario,
  activo
FROM read_files(
  '/Volumes/workspace/sql_practice_raw/datasets/caso_01_ecommerce_andino/productos.txt',
  format => 'csv',
  header => true,
  mode => 'FAILFAST',
  schema => 'producto_id STRING, nombre_producto STRING, categoria STRING, marca STRING, precio_lista DECIMAL(12,2), costo_unitario DECIMAL(12,2), activo BOOLEAN'
);

SELECT * FROM productos LIMIT 5;

producto_id,nombre_producto,categoria,marca,precio_lista,costo_unitario,activo
P0001,Laptop Nova 14,Tecnología,NovaTech,2130.00,1295.55,true
P0002,Mouse Ergonómico,Tecnología,ClickPro,1095.00,590.93,true
P0003,Teclado Mecánico,Tecnología,KeyMax,2220.00,1335.93,true
P0004,Audífonos Bluetooth,Tecnología,SoundGo,735.00,518.89,true
P0005,Monitor 24 pulgadas,Tecnología,ViewPlus,2715.00,1746.64,false


#### `pedidos`

In [0]:
%sql
INSERT INTO pedidos
SELECT
  pedido_id,
  cliente_id,
  vendedor_id,
  fecha_pedido,
  canal,
  metodo_pago,
  estado,
  ciudad_envio,
  fecha_entrega
FROM read_files(
  '/Volumes/workspace/sql_practice_raw/datasets/caso_01_ecommerce_andino/pedidos.txt',
  format => 'csv',
  header => true,
  mode => 'FAILFAST',
  schema => 'pedido_id STRING, cliente_id STRING, vendedor_id STRING, fecha_pedido DATE, canal STRING, metodo_pago STRING, estado STRING, ciudad_envio STRING, fecha_entrega DATE'
);

SELECT * FROM pedidos LIMIT 5;

pedido_id,cliente_id,vendedor_id,fecha_pedido,canal,metodo_pago,estado,ciudad_envio,fecha_entrega
O00001,C0037,V002,2025-08-13,App,Tarjeta,entregado,Cajamarca,2025-08-24
O00002,C0092,V007,2026-06-19,App,Plin,pendiente,Ayacucho,null
O00003,C0035,V012,2025-09-24,Tienda,Tarjeta,entregado,Puno,2025-09-30
O00004,C0117,V008,2026-05-11,Web,Tarjeta,entregado,Cusco,2026-05-16
O00005,C0057,V010,2025-09-16,App,Efectivo,entregado,Trujillo,2025-09-23


#### `detalle_pedidos`

In [0]:
%sql
INSERT INTO detalle_pedidos
SELECT
  pedido_id,
  linea_id,
  producto_id,
  cantidad,
  precio_unitario,
  descuento_pct
FROM read_files(
  '/Volumes/workspace/sql_practice_raw/datasets/caso_01_ecommerce_andino/detalle_pedidos.txt',
  format => 'csv',
  header => true,
  mode => 'FAILFAST',
  schema => 'pedido_id STRING, linea_id INT, producto_id STRING, cantidad INT, precio_unitario DECIMAL(12,2), descuento_pct DECIMAL(5,2)'
);

SELECT * FROM detalle_pedidos LIMIT 5;

pedido_id,linea_id,producto_id,cantidad,precio_unitario,descuento_pct
O00001,1,P0034,1,136.57,10.00
O00001,2,P0024,1,88.29,15.00
O00001,3,P0012,1,167.53,25.00
O00001,4,P0031,2,110.51,0.00
O00002,1,P0004,2,708.06,10.00


#### `devoluciones`

In [0]:
%sql
INSERT INTO devoluciones
SELECT
  devolucion_id,
  pedido_id,
  producto_id,
  fecha_devolucion,
  cantidad_devuelta,
  motivo,
  monto_reembolso
FROM read_files(
  '/Volumes/workspace/sql_practice_raw/datasets/caso_01_ecommerce_andino/devoluciones.txt',
  format => 'csv',
  header => true,
  mode => 'FAILFAST',
  schema => 'devolucion_id STRING, pedido_id STRING, producto_id STRING, fecha_devolucion DATE, cantidad_devuelta INT, motivo STRING, monto_reembolso DECIMAL(12,2)'
);

SELECT * FROM devoluciones LIMIT 5;

devolucion_id,pedido_id,producto_id,fecha_devolucion,cantidad_devuelta,motivo,monto_reembolso
R0001,O00001,P0034,2025-09-04,1,Talla incorrecta,122.91
R0002,O00006,P0039,2026-01-26,1,Talla incorrecta,780.62
R0003,O00012,P0015,2025-12-03,3,No era lo esperado,800.70
R0004,O00029,P0005,2025-11-17,1,Talla incorrecta,2175.86
R0005,O00055,P0006,2025-08-23,1,Talla incorrecta,2018.57


### 0.6 Validación

Todas las tablas deben mostrar `estado = OK`.

In [0]:
%sql
WITH conteos AS (
  SELECT 'clientes' AS tabla, COUNT(*) AS filas, 120 AS filas_esperadas FROM clientes
  UNION ALL
  SELECT 'vendedores' AS tabla, COUNT(*) AS filas, 12 AS filas_esperadas FROM vendedores
  UNION ALL
  SELECT 'productos' AS tabla, COUNT(*) AS filas, 45 AS filas_esperadas FROM productos
  UNION ALL
  SELECT 'pedidos' AS tabla, COUNT(*) AS filas, 360 AS filas_esperadas FROM pedidos
  UNION ALL
  SELECT 'detalle_pedidos' AS tabla, COUNT(*) AS filas, 1030 AS filas_esperadas FROM detalle_pedidos
  UNION ALL
  SELECT 'devoluciones' AS tabla, COUNT(*) AS filas, 63 AS filas_esperadas FROM devoluciones
)
SELECT
  tabla,
  filas,
  filas_esperadas,
  IF(filas = filas_esperadas, 'OK', 'REVISAR') AS estado
FROM conteos;

tabla,filas,filas_esperadas,estado
clientes,120,120,OK
vendedores,12,12,OK
productos,45,45,OK
pedidos,360,360,OK
detalle_pedidos,1030,1030,OK
devoluciones,63,63,OK


Integridad referencial: como las claves foráneas son informativas, comprobamos que no existan filas hijas que apunten a un padre inexistente. Todas las relaciones deben mostrar `huerfanos = 0`.

In [0]:
%sql
SELECT 'pedidos.cliente_id -> clientes' AS relacion, COUNT(*) AS huerfanos
FROM pedidos AS h
LEFT JOIN clientes AS p ON h.cliente_id = p.cliente_id
WHERE p.cliente_id IS NULL
UNION ALL
SELECT 'pedidos.vendedor_id -> vendedores' AS relacion, COUNT(*) AS huerfanos
FROM pedidos AS h
LEFT JOIN vendedores AS p ON h.vendedor_id = p.vendedor_id
WHERE p.vendedor_id IS NULL
UNION ALL
SELECT 'detalle_pedidos.pedido_id -> pedidos' AS relacion, COUNT(*) AS huerfanos
FROM detalle_pedidos AS h
LEFT JOIN pedidos AS p ON h.pedido_id = p.pedido_id
WHERE p.pedido_id IS NULL
UNION ALL
SELECT 'detalle_pedidos.producto_id -> productos' AS relacion, COUNT(*) AS huerfanos
FROM detalle_pedidos AS h
LEFT JOIN productos AS p ON h.producto_id = p.producto_id
WHERE p.producto_id IS NULL
UNION ALL
SELECT 'devoluciones.pedido_id -> pedidos' AS relacion, COUNT(*) AS huerfanos
FROM devoluciones AS h
LEFT JOIN pedidos AS p ON h.pedido_id = p.pedido_id
WHERE p.pedido_id IS NULL
UNION ALL
SELECT 'devoluciones.producto_id -> productos' AS relacion, COUNT(*) AS huerfanos
FROM devoluciones AS h
LEFT JOIN productos AS p ON h.producto_id = p.producto_id
WHERE p.producto_id IS NULL;

relacion,huerfanos
pedidos.cliente_id -> clientes,0
pedidos.vendedor_id -> vendedores,0
detalle_pedidos.pedido_id -> pedidos,0
detalle_pedidos.producto_id -> productos,0
devoluciones.pedido_id -> pedidos,0
devoluciones.producto_id -> productos,0


---
## Configuración para resolver las preguntas

Ejecuta esta celda cada vez que abras el notebook: las consultas usan nombres de tablas sin catálogo ni esquema.

In [0]:
%sql
USE CATALOG workspace;
USE SCHEMA sql_practice_ecommerce;

SHOW TABLES;

database,tableName,isTemporary
sql_practice_ecommerce,clientes,false
sql_practice_ecommerce,detalle_pedidos,false
sql_practice_ecommerce,devoluciones,false
sql_practice_ecommerce,pedidos,false
sql_practice_ecommerce,productos,false
sql_practice_ecommerce,vendedores,false


---
## PARTE 1 — Preguntas básicas (1-10)

### 1. Todos los clientes

Muestra todos los registros de la tabla `clientes`.

In [0]:
%sql
-- Escribe tu consulta aquí
SELECT * FROM clientes;

cliente_id,nombre_cliente,ciudad,region,segmento,fecha_registro,email
C0001,Renzo Reyes,Cusco,Sur,Persona,2024-03-06,renzo.reyes1@example.com
C0002,Natalia Aguilar,Piura,Norte,Persona,2024-12-04,natalia.aguilar2@example.com
C0003,Hugo Núñez,Cajamarca,Norte,Empresa,2024-10-12,hugo.nunez3@example.com
C0004,Carla Mendoza,Cusco,Sur,Persona,2025-08-26,carla.mendoza4@example.com
C0005,Carlos Silva,Puno,Sur,Persona,2024-01-19,carlos.silva5@example.com
C0006,Bruno Ortega,Ica,Costa,Persona,2025-11-07,bruno.ortega6@example.com
C0007,Daniela Navarro,Huancayo,Centro,Persona,2024-03-02,daniela.navarro7@example.com
C0008,Gabriela Acosta,Arequipa,Sur,Persona,2025-10-27,gabriela.acosta8@example.com
C0009,Miguel Paredes,Lima,Costa,Persona,2025-11-10,miguel.paredes9@example.com
C0010,Jorge Reyes,Tacna,Sur,Pyme,2025-08-26,jorge.reyes10@example.com


### 2. Productos activos

Lista el ID, nombre, categoría y precio de los productos activos.

In [0]:
%sql
-- Escribe tu consulta aquí
SELECT producto_id, nombre_producto, categoria, precio_lista FROM productos WHERE activo = true

producto_id,nombre_producto,categoria,precio_lista
P0001,Laptop Nova 14,Tecnología,2130.00
P0002,Mouse Ergonómico,Tecnología,1095.00
P0003,Teclado Mecánico,Tecnología,2220.00
P0004,Audífonos Bluetooth,Tecnología,735.00
P0006,Webcam Full HD,Tecnología,2135.00
P0007,SSD Externo 1TB,Tecnología,205.00
P0008,Smartwatch Active,Tecnología,1855.00
P0010,Cafetera Digital,Hogar,145.00
P0011,Freidora de Aire,Hogar,690.00
P0012,Juego de Sábanas,Hogar,165.00


### 3. Pedidos entregados

Muestra los pedidos cuyo estado sea `entregado`.

In [0]:
%sql
-- Escribe tu consulta aquí
SELECT * FROM pedidos WHERE estado = 'entregado'

pedido_id,cliente_id,vendedor_id,fecha_pedido,canal,metodo_pago,estado,ciudad_envio,fecha_entrega
O00001,C0037,V002,2025-08-13,App,Tarjeta,entregado,Cajamarca,2025-08-24
O00003,C0035,V012,2025-09-24,Tienda,Tarjeta,entregado,Puno,2025-09-30
O00004,C0117,V008,2026-05-11,Web,Tarjeta,entregado,Cusco,2026-05-16
O00005,C0057,V010,2025-09-16,App,Efectivo,entregado,Trujillo,2025-09-23
O00006,C0006,V001,2026-01-07,Web,Plin,entregado,Ica,2026-01-14
O00007,C0039,V010,2025-12-08,Web,Tarjeta,entregado,Trujillo,2025-12-18
O00009,C0099,V007,2026-01-11,Web,Transferencia,entregado,Ayacucho,2026-01-14
O00010,C0037,V006,2025-02-13,App,Tarjeta,entregado,Cajamarca,2025-02-15
O00012,C0111,V007,2025-11-28,Web,Transferencia,entregado,Huancayo,2025-12-01
O00013,C0034,V006,2025-02-24,App,Yape,entregado,Trujillo,2025-02-27


### 4. Ciudades únicas de clientes

Obtén la lista de ciudades únicas de los clientes, ordenadas alfabéticamente.

In [0]:
%sql
-- Escribe tu consulta aquí
SELECT DISTINCT ciudad as ciudad_unica FROM clientes

ciudad_unica
Cusco
Piura
Cajamarca
Puno
Ica
Huancayo
Arequipa
Lima
Tacna
Chiclayo


### 5. Diez productos con mayor precio

Muestra los 10 productos con mayor precio de lista.

In [0]:
%sql
-- Escribe tu consulta aquí
SELECT * FROM productos ORDER BY precio_lista DESC LIMIT 10

producto_id,nombre_producto,categoria,marca,precio_lista,costo_unitario,activo
P0005,Monitor 24 pulgadas,Tecnología,ViewPlus,2715.00,1746.64,false
P0003,Teclado Mecánico,Tecnología,KeyMax,2220.00,1335.93,true
P0006,Webcam Full HD,Tecnología,VisionX,2135.00,1434.37,true
P0001,Laptop Nova 14,Tecnología,NovaTech,2130.00,1295.55,true
P0008,Smartwatch Active,Tecnología,Pulse,1855.00,959.23,true
P0002,Mouse Ergonómico,Tecnología,ClickPro,1095.00,590.93,true
P0044,Pizarra Magnética,Oficina,IdeaBoard,940.00,590.83,true
P0039,Hub USB-C,Oficina,LinkPro,940.00,607.02,true
P0040,Soporte para Laptop,Oficina,DeskUp,890.00,582.23,true
P0004,Audífonos Bluetooth,Tecnología,SoundGo,735.00,518.89,true


### 6. Cantidad total de pedidos

Cuenta cuántos pedidos existen en total.

In [0]:
%sql
-- Escribe tu consulta aquí
SELECT COUNT(1) FROM pedidos

COUNT(1)
360


### 7. Precio promedio de los productos

Calcula el precio de lista promedio de todos los productos.

In [0]:
%sql
-- Escribe tu consulta aquí
SELECT AVG(precio_lista) FROM productos

AVG(precio_lista)
543.222222


### 8. Pedidos de enero de 2026

Muestra los pedidos realizados durante enero de 2026.

In [0]:
%sql
-- Escribe tu consulta aquí
SELECT * FROM pedidos WHERE fecha_pedido >= '2026-01-01' AND fecha_pedido <= '2026-01-31' ORDER BY fecha_pedido ASC

pedido_id,cliente_id,vendedor_id,fecha_pedido,canal,metodo_pago,estado,ciudad_envio,fecha_entrega
O00168,C0041,V006,2026-01-01,App,Efectivo,entregado,Chiclayo,2026-01-05
O00284,C0117,V004,2026-01-01,Web,Yape,entregado,Cusco,2026-01-08
O00066,C0091,V002,2026-01-02,Web,Yape,entregado,Chiclayo,2026-01-09
O00314,C0012,V004,2026-01-02,App,Tarjeta,enviado,Arequipa,null
O00064,C0083,V002,2026-01-02,Web,Transferencia,pendiente,Piura,null
O00205,C0011,V009,2026-01-03,Web,Plin,enviado,Lima,null
O00173,C0004,V004,2026-01-04,Web,Tarjeta,cancelado,Cusco,null
O00279,C0084,V011,2026-01-07,App,Plin,entregado,Ayacucho,2026-01-11
O00006,C0006,V001,2026-01-07,Web,Plin,entregado,Ica,2026-01-14
O00244,C0040,V010,2026-01-09,App,Transferencia,entregado,Piura,2026-01-10


### 9. Clientes por segmento

Cuenta cuántos clientes existen por segmento.

In [0]:
%sql
-- Escribe tu consulta aquí
SELECT segmento, COUNT(1) AS cantidad_cliente FROM clientes GROUP BY segmento

segmento,cantidad_cliente
Persona,76
Empresa,10
Pyme,34


### 10. Importe neto por línea

Muestra cada línea de pedido incluyendo una columna calculada `importe_neto_linea`.

In [0]:
%sql
-- Escribe tu consulta aquí
SELECT 
    DISTINCT(linea_id) AS linea,
    ROUND (SUM(cantidad * precio_unitario * (1 - descuento_pct / 100)),2) AS importe_neto_en_linea
FROM detalle_pedidos
GROUP BY linea_id

linea,importe_neto_en_linea
1,309226.47
2,309413.81
3,133630.66
4,115862.63
5,47923.93


---
## PARTE 2 — Preguntas intermedias (11-30)

### 11. Clientes con cantidad de pedidos, incluyendo clientes sin pedidos

Muestra cada cliente con su cantidad total de pedidos, incluyendo clientes sin pedidos.

In [0]:
%sql
-- Escribe tu consulta aquí
SELECT 
    c.cliente_id,
    c.nombre_cliente,
    COUNT(p.cliente_id) AS cantidad_pedidos
FROM clientes c
LEFT JOIN pedidos p ON c.cliente_id = p.cliente_id
GROUP BY
c.cliente_id, c.nombre_cliente
ORDER BY cantidad_pedidos DESC

cliente_id,nombre_cliente,cantidad_pedidos
C0019,Daniela Peña,9
C0096,Gabriela Silva,8
C0011,Álvaro Silva,7
C0065,Camila Castro,7
C0002,Natalia Aguilar,7
C0076,Mateo Vargas,6
C0034,Alejandra Aguilar,6
C0117,Andrea Reyes,6
C0090,Camila Medina,6
C0004,Carla Mendoza,6


### 12. Ventas netas válidas por categoría

Calcula las ventas netas válidas por categoría de producto.

In [0]:
%sql
-- Escribe tu consulta aquí
SELECT 
    p.categoria,
    ROUND(SUM(d.cantidad * d.precio_unitario * (1 - d.descuento_pct / 100)),2) AS ventas_netas_validadas
FROM detalle_pedidos d
LEFT JOIN productos p ON d.producto_id = p.producto_id
LEFT JOIN pedidos pe ON d.pedido_id = pe.pedido_id 
WHERE pe.estado != 'cancelado'
GROUP BY p.categoria
ORDER BY ventas_netas_validadas DESC

categoria,ventas_netas_validadas
Tecnología,484789.88
Oficina,125504.25
Hogar,107762.25
Deportes,46057.55
Belleza,35519.96
Moda,24510.65


### 13. Ticket promedio de pedidos entregados

Calcula el ticket promedio de los pedidos entregados. Primero suma cada pedido y luego promedia esos totales.

In [0]:
%sql
-- Escribe tu consulta aquí
WITH total_de_pedidos AS (
    SELECT 
        p.pedido_id AS pedido,
        ROUND(
            SUM(
                d.cantidad * d.precio_unitario * (1 - d.descuento_pct / 100)
                ),
        2) AS suma_pedido
    FROM pedidos p
    LEFT JOIN detalle_pedidos d ON d.pedido_id = p.pedido_id 
    WHERE p.estado = 'entregado'
    GROUP BY p.pedido_id 
)
SELECT
    ROUND(AVG(suma_pedido),2) AS ticket_prom_entregados
FROM total_de_pedidos

ticket_prom_entregados
2571.05


### 14. Pedidos válidos y ventas por mes

Obtén la cantidad de pedidos válidos y las ventas netas por mes.

In [0]:
%sql
-- Escribe tu consulta aquí
SELECT 
    DATE_FORMAT(p.fecha_pedido, 'yyyyMM') AS mes,
    COUNT(DISTINCT p.pedido_id) AS cant_pedidos_validos,
    SUM(d.cantidad * d.precio_unitario * (1 - d.descuento_pct / 100)) AS venta_neta
FROM pedidos  p
INNER JOIN detalle_pedidos  d ON p.pedido_id = d.pedido_id
WHERE p.estado != 'cancelado'
GROUP BY DATE_FORMAT(p.fecha_pedido, 'yyyyMM')
ORDER BY mes

mes,cant_pedidos_validos,venta_neta
202501,10,30543.70050000
202502,11,37437.39050000
202503,6,14321.27200000
202504,10,29019.87100000
202505,8,16668.40250000
202506,9,21692.88900000
202507,17,48426.42950000
202508,18,45525.97850000
202509,16,46879.12600000
202510,18,52283.78800000


### 15. Top 10 clientes por gasto neto

Identifica los 10 clientes con mayor gasto neto en pedidos válidos.

In [0]:
%sql
-- Escribe tu consulta aquí

SELECT 
    c.cliente_id,
    c.nombre_cliente,
    --pedidos.estado,
    SUM(
        dp.cantidad * dp.precio_unitario * (1 - dp.descuento_pct / 100)
    ) AS gasto_neto_cliente
FROM clientes c
INNER JOIN pedidos p ON c.cliente_id = p.cliente_id
INNER JOIN detalle_pedidos dp ON p.pedido_id = dp.pedido_id
WHERE p.estado != 'cancelado'
GROUP BY c.cliente_id, c.nombre_cliente
ORDER BY gasto_neto_cliente DESC
LIMIT 10

cliente_id,nombre_cliente,gasto_neto_cliente
C0027,Ana Valdez,31382.79700000
C0090,Camila Medina,29452.63000000
C0019,Daniela Peña,26312.57900000
C0096,Gabriela Silva,23669.58500000
C0098,María Acosta,22783.13950000
C0011,Álvaro Silva,21690.86850000
C0034,Alejandra Aguilar,20064.62450000
C0101,Marco Campos,19790.04600000
C0087,Carla Ramírez,16024.25850000
C0076,Mateo Vargas,15415.04600000


### 16. Productos nunca vendidos

Lista los productos que nunca aparecen en `detalle_pedidos`.

In [0]:
%sql
-- Escribe tu consulta aquí
SELECT 
    p.producto_id,
    p.nombre_producto
FROM productos  p
LEFT ANTI JOIN detalle_pedidos  dp ON p.producto_id = dp.producto_id

producto_id,nombre_producto
P0043,Calculadora Científica
P0044,Pizarra Magnética
P0045,Archivador A4


### 17. Clientes sin pedidos

Lista los clientes que nunca realizaron un pedido.

In [0]:
%sql
-- Escribe tu consulta aquí
SELECT
    c.cliente_id,
    c.nombre_cliente
FROM clientes c
LEFT ANTI JOIN pedidos p ON c.cliente_id = p.cliente_id

cliente_id,nombre_cliente
C0029,Carla Reyes
C0051,Marco Mendoza
C0055,Jorge Rojas
C0066,Sofía Sánchez
C0075,María Espinoza
C0081,Fernando Herrera
C0088,Andrea Castro
C0113,Natalia Cruz


### 18. Promedio de días de entrega por canal

Calcula el promedio de días de entrega por canal para pedidos entregados.

In [0]:
%sql
-- Escribe tu consulta aquí
WITH pedidos_entregados AS (
    SELECT
        canal,
        DATEDIFF(fecha_entrega, fecha_pedido) AS dias_entrega
    FROM pedidos
    WHERE estado = 'entregado'
)
SELECT
    canal,
    ROUND(AVG(dias_entrega), 2) AS promedio_dias_entrega
FROM pedidos_entregados
GROUP BY canal

canal,promedio_dias_entrega
App,5.96
Tienda,5.42
Web,6.59


### 19. Unidades vendidas, devueltas y porcentaje de devolución por categoría

Calcula por categoría: unidades vendidas, unidades devueltas y porcentaje de devolución sobre unidades vendidas. Considera ventas válidas.

In [0]:
%sql
-- Escribe tu consulta aquí
WITH ventas_por_categoria AS (
    SELECT
        pr.categoria,
        SUM(dp.cantidad) AS nro_vendidas
    FROM detalle_pedidos dp
    INNER JOIN pedidos p ON dp.pedido_id = p.pedido_id
    LEFT JOIN productos pr  ON dp.producto_id = pr.producto_id
    WHERE p.estado != 'cancelado'
    GROUP BY pr.categoria
),
devoluciones_categoria AS (
    SELECT
        p.categoria,
        SUM(d.cantidad_devuelta) AS nro_devueltas
    FROM devoluciones d
    LEFT JOIN productos p ON d.producto_id = p.producto_id
    GROUP BY p.categoria
)
SELECT
    v.categoria,
    v.nro_vendidas,
    COALESCE(d.nro_devueltas, 0) AS unidades_devueltas,
    ROUND(
        COALESCE(d.nro_devueltas, 0) * 100.0
        / NULLIF(v.nro_vendidas, 0),
        2
    ) AS porcentaje_de_devolucion
FROM ventas_por_categoria AS v
LEFT JOIN devoluciones_categoria AS d
    ON v.categoria = d.categoria
ORDER BY porcentaje_de_devolucion DESC

categoria,nro_vendidas,unidades_devueltas,porcentaje_de_devolucion
Tecnología,321,29,9.03
Hogar,275,17,6.18
Deportes,265,16,6.04
Oficina,217,8,3.69
Belleza,292,9,3.08
Moda,311,7,2.25


### 20. Ventas mensuales por vendedor y cumplimiento de meta

Calcula las ventas mensuales por vendedor y el porcentaje de cumplimiento de su meta mensual.

In [0]:
%sql
-- Escribe tu consulta aquí

WITH venta_mensual_vendedor AS (
    SELECT 
        date_format(p.fecha_pedido, 'yyyy-MM') AS mes,
        v.vendedor_id, 
        v.nombre_vendedor,
       ROUND(SUM (dp.cantidad * dp.precio_unitario * (1 - dp.descuento_pct / 100)),2) AS ventas
    FROM vendedores v
    LEFT JOIN pedidos p ON v.vendedor_id = p.vendedor_id
    LEFT JOIN detalle_pedidos dp ON p.pedido_id = dp.pedido_id
    WHERE p.estado !='cancelado'
    GROUP BY v.vendedor_id, v.nombre_vendedor, mes
    ORDER BY mes, ventas
)
SELECT 
    vm.*,v.meta_mensual, 
    round(vm.ventas*100/v.meta_mensual,2) AS pct_porcentaje
FROM venta_mensual_vendedor vm
LEFT JOIN vendedores v ON vm.vendedor_id = v.vendedor_id
ORDER BY mes, ventas

mes,vendedor_id,nombre_vendedor,ventas,meta_mensual,pct_porcentaje
2025-01,V009,Carlos Aguilar,1184.61,30000.00,3.95
2025-01,V006,Luis Rojas,2034.42,12000.00,16.95
2025-01,V005,Paola Silva,4255.14,30000.00,14.18
2025-01,V010,Renzo Salazar,4388.99,18000.00,24.38
2025-01,V008,Diego Navarro,4725.54,12000.00,39.38
2025-01,V001,José Cárdenas,5161.86,25000.00,20.65
2025-01,V012,Sebastián Herrera,8793.15,25000.00,35.17
2025-02,V009,Carlos Aguilar,192.91,30000.00,0.64
2025-02,V011,Álvaro López,578.76,30000.00,1.93
2025-02,V004,Camila Herrera,1972.51,18000.00,10.96


### 21. Ranking de productos por ventas dentro de cada categoría

Genera un ranking de productos por ventas netas dentro de cada categoría.

In [0]:
%sql
-- Escribe tu consulta aquí

WITH ventas_producto AS (
    SELECT
        p.categoria,
        p.producto_id,
        p.nombre_producto,
        ROUND(
            SUM(
                d.cantidad * d.precio_unitario
                * (1 - COALESCE(d.descuento_pct, 0) / 100.0)
            ),
            2
        ) AS venta_realizadas
    FROM productos p
    INNER JOIN detalle_pedidos d ON d.producto_id = p.producto_id
    INNER JOIN pedidos pe ON pe.pedido_id = d.pedido_id
    WHERE pe.estado != 'cancelado'
    GROUP BY
        p.categoria, p.producto_id, p.nombre_producto
)
SELECT
    categoria,
    producto_id,
    nombre_producto,
    venta_realizadas,
    DENSE_RANK() OVER (
        PARTITION BY categoria
        ORDER BY venta_realizadas DESC
    ) AS ranking_ventas
FROM ventas_producto
ORDER BY
    categoria, ranking_ventas, nombre_producto


categoria,producto_id,nombre_producto,venta_realizadas,ranking_ventas
Belleza,P0034,Crema Hidratante,6832.88,1
Belleza,P0035,Perfume Floral,6760.54,2
Belleza,P0030,Protector Solar,6104.95,3
Belleza,P0036,Shampoo Natural,5513.51,4
Belleza,P0031,Serum Facial,5154.62,5
Belleza,P0033,Kit de Brochas,3420.58,6
Belleza,P0032,Secadora Compacta,1732.89,7
Deportes,P0028,Guantes Fitness,11812.62,1
Deportes,P0023,Mat de Yoga,8294.42,2
Deportes,P0026,Banda Elástica,7300.20,3


### 22. Ventas mensuales y acumulado

Calcula las ventas netas mensuales y el acumulado de ventas desde el primer mes.

In [0]:
%sql
-- Escribe tu consulta aquí


WITH ventas_mensuales AS (
    SELECT
        DATE_FORMAT(p.fecha_pedido, 'yyyy-MM') AS mes,
        ROUND(
            SUM(
                dp.cantidad * dp.precio_unitario
                * (1 - COALESCE(dp.descuento_pct, 0) / 100.0)
            ),
            2
        ) AS ventas_realizadas
    FROM detalle_pedidos dp
    INNER JOIN pedidos p ON p.pedido_id = dp.pedido_id
    WHERE p.estado != 'cancelado'
    GROUP BY DATE_FORMAT(p.fecha_pedido, 'yyyy-MM')
)
SELECT
    mes,
    ventas_realizadas,
    ROUND(
        SUM(ventas_realizadas) OVER (
            ORDER BY mes
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ),
        2
    ) AS ventas_acum
FROM ventas_mensuales
ORDER BY mes


mes,ventas_realizadas,ventas_acum
2025-01,30543.70,30543.70
2025-02,37437.39,67981.09
2025-03,14321.27,82302.36
2025-04,29019.87,111322.23
2025-05,16668.40,127990.63
2025-06,21692.89,149683.52
2025-07,48426.43,198109.95
2025-08,45525.98,243635.93
2025-09,46879.13,290515.06
2025-10,52283.79,342798.85


### 23. Crecimiento porcentual respecto al mes anterior

Calcula el crecimiento porcentual de ventas netas respecto al mes anterior.

In [0]:
%sql
-- Escribe tu consulta aquí


WITH ventas_mensual AS (
    SELECT
        DATE_FORMAT(p.fecha_pedido, 'yyyy-MM')AS mes,
        SUM(
            d.cantidad * d.precio_unitario
            * (1 - COALESCE(d.descuento_pct, 0) / 100.0)
        ) AS ventas_netas
    FROM pedidos p
    INNER JOIN detalle_pedidos d ON d.pedido_id = p.pedido_id
    WHERE p.estado != 'cancelado'
    GROUP BY DATE_FORMAT(p.fecha_pedido, 'yyyy-MM')
),
crecimiento AS (
    SELECT
        mes,
        ventas_netas,
        LAG(ventas_netas) OVER (
            ORDER BY mes
        ) AS ventas_mes_anterior
    FROM ventas_mensual
)
SELECT
    mes,
    ROUND(ventas_netas, 2) AS ventas_netas_total,
    ROUND(ventas_mes_anterior, 2) AS ventas_mes_netas_anterior,
    ROUND(
        (ventas_netas - ventas_mes_anterior) * 100.0
        / NULLIF(ventas_mes_anterior, 0),
        2
    ) AS pct_ventas_crecimiento
FROM crecimiento
ORDER BY mes



mes,ventas_netas_total,ventas_mes_netas_anterior,pct_ventas_crecimiento
2025-01,30543.70,null,null
2025-02,37437.39,30543.70,22.57
2025-03,14321.27,37437.39,-61.75
2025-04,29019.87,14321.27,102.63
2025-05,16668.40,29019.87,-42.56
2025-06,21692.89,16668.40,30.14
2025-07,48426.43,21692.89,123.24
2025-08,45525.98,48426.43,-5.99
2025-09,46879.13,45525.98,2.97
2025-10,52283.79,46879.13,11.53


### 24. Primera y última compra por cliente

Para cada cliente con pedidos, muestra la fecha de su primera compra y la fecha de su compra más reciente.

In [0]:
%sql
-- Escribe tu consulta aquí
SELECT
    c.cliente_id,
    c.nombre_cliente,
    MIN(p.fecha_pedido) AS primera_compra,
    MAX(p.fecha_pedido) AS ultima_compra
FROM clientes c
INNER JOIN pedidos p ON p.cliente_id = c.cliente_id
GROUP BY
    c.cliente_id, c.nombre_cliente
ORDER BY
    c.cliente_id


cliente_id,nombre_cliente,primera_compra,ultima_compra
C0001,Renzo Reyes,2025-04-10,2025-04-10
C0002,Natalia Aguilar,2025-03-06,2026-01-30
C0003,Hugo Núñez,2025-09-14,2026-05-12
C0004,Carla Mendoza,2026-01-04,2026-06-29
C0005,Carlos Silva,2025-10-20,2025-12-22
C0006,Bruno Ortega,2025-11-16,2026-06-26
C0007,Daniela Navarro,2025-06-02,2026-02-22
C0008,Gabriela Acosta,2025-12-18,2026-06-28
C0009,Miguel Paredes,2025-12-24,2026-04-10
C0010,Jorge Reyes,2025-08-29,2025-10-16


### 25. Clientes con compras válidas en al menos 3 meses distintos

Lista los clientes que realizaron pedidos válidos en al menos 3 meses distintos.

In [0]:
%sql
-- Escribe tu consulta aquí
WITH lista_clientes AS (
SELECT
    c.cliente_id,
    c.nombre_cliente,
    COUNT(
        DISTINCT DATE_FORMAT(p.fecha_pedido, 'yyyy-MM')
    ) AS meses_diferentes
FROM clientes c
INNER JOIN pedidos p ON p.cliente_id = c.cliente_id
WHERE p.estado != 'cancelado'
GROUP BY
    c.cliente_id, c.nombre_cliente
)
SELECT 
    cliente_id,
    nombre_cliente
FROM 
    lista_clientes
WHERE
    meses_diferentes >= 3

cliente_id,nombre_cliente
C0115,Sofía Torres
C0107,Marco Medina
C0076,Mateo Vargas
C0011,Álvaro Silva
C0003,Hugo Núñez
C0101,Marco Campos
C0119,Paola Medina
C0065,Camila Castro
C0008,Gabriela Acosta
C0069,Miguel Ortega


### 26. Pedidos con importe superior al promedio

Muestra los pedidos válidos cuyo importe total sea superior al importe promedio de todos los pedidos válidos.

In [0]:
%sql
-- Escribe tu consulta aquí
WITH importe_de_pedido AS (
    SELECT
        p.pedido_id,
        ROUND(
            SUM(
                d.cantidad * d.precio_unitario
                * (1 - COALESCE(d.descuento_pct, 0) / 100.0)
            ), 2
        ) AS ventas_netas
    FROM pedidos p
    INNER JOIN detalle_pedidos d ON d.pedido_id = p.pedido_id
    WHERE p.estado != 'cancelado'
    GROUP BY p.pedido_id
),
pedidos_promedio AS (
    SELECT
        pedido_id,
        ventas_netas,
        AVG(ventas_netas) OVER () AS promedio_de_ventas
    FROM importe_de_pedido
)
SELECT
    pedido_id,
    ventas_netas
FROM pedidos_promedio
WHERE ventas_netas > promedio_de_ventas
ORDER BY ventas_netas DESC

pedido_id,ventas_netas
O00156,14110.78
O00166,10814.88
O00259,10431.39
O00342,10417.27
O00096,10195.43
O00344,9540.53
O00012,9290.06
O00029,9285.18
O00150,9139.40
O00100,8932.32


### 27. Productos con precio superior al promedio de su categoría

Lista los productos cuyo precio de lista sea superior al precio promedio de su propia categoría.

In [0]:
%sql
-- Escribe tu consulta aquí

WITH productos_con_promedio AS (
    SELECT
        producto_id,
        nombre_producto,
        categoria,
        precio_lista,
        AVG(precio_lista) OVER (
            PARTITION BY categoria
        ) AS promedio_categoria
    FROM productos
),
productos_superiores AS (
    SELECT
        producto_id,
        nombre_producto,
        categoria,
        precio_lista,
        promedio_categoria
    FROM productos_con_promedio
    WHERE precio_lista > promedio_categoria
)
SELECT *
FROM productos_superiores
ORDER BY
    categoria, precio_lista DESC


producto_id,nombre_producto,categoria,precio_lista,promedio_categoria
P0033,Kit de Brochas,Belleza,185.00,135.000000
P0035,Perfume Floral,Belleza,165.00,135.000000
P0030,Protector Solar,Belleza,155.00,135.000000
P0027,Botella Térmica,Deportes,255.00,190.714286
P0026,Banda Elástica,Deportes,250.00,190.714286
P0028,Guantes Fitness,Deportes,245.00,190.714286
P0023,Mat de Yoga,Deportes,215.00,190.714286
P0011,Freidora de Aire,Hogar,690.00,425.714286
P0013,Organizador Modular,Hogar,685.00,425.714286
P0014,Lámpara LED,Hogar,540.00,425.714286


### 28. Participación de cada categoría en las ventas totales

Calcula el porcentaje de participación de cada categoría sobre las ventas netas válidas totales.

In [0]:
%sql
-- Escribe tu consulta aquí


WITH ventas_por_categoria AS (
    SELECT
        pr.categoria,
        SUM(
            dp.cantidad * dp.precio_unitario
            * (1 - COALESCE(dp.descuento_pct, 0) / 100.0)
        ) AS ventas_netas
    FROM detalle_pedidos dp
    INNER JOIN pedidos  p ON p.pedido_id = dp.pedido_id
    INNER JOIN productos AS pr ON pr.producto_id = dp.producto_id
    WHERE p.estado <> 'cancelado'
    GROUP BY pr.categoria
),
participacion AS (
    SELECT
        categoria,
        ventas_netas,
        SUM(ventas_netas) OVER () AS venta_total
    FROM ventas_por_categoria
)
SELECT
    categoria,
    ROUND(ventas_netas, 2) AS ventas_netas_categoria,
    ROUND(venta_total, 2) AS ventas_netas_totales,
    ROUND(
        100.0 * ventas_netas / NULLIF(venta_total, 0),
        2
    ) AS pct_participacion
FROM participacion
ORDER BY pct_participacion DESC;


categoria,ventas_netas_categoria,ventas_netas_totales,pct_participacion
Tecnología,484789.88,824144.52,58.82
Oficina,125504.25,824144.52,15.23
Hogar,107762.25,824144.52,13.08
Deportes,46057.55,824144.52,5.59
Belleza,35519.96,824144.52,4.31
Moda,24510.65,824144.52,2.97


### 29. Método de pago más usado por ciudad de envío

Identifica el método de pago más utilizado en cada ciudad de envío. Si hay empate, elige alfabéticamente el método de pago.

In [0]:
%sql
-- Escribe tu consulta aquí
WITH conteo_pagos AS (
    SELECT
        ciudad_envio,
        metodo_pago,
        COUNT(*) AS total_uso
    FROM pedidos
    WHERE estado != 'cancelado'
      AND metodo_pago IS NOT NULL
    GROUP BY
        ciudad_envio, metodo_pago
),
maximos_ciudad AS (
    SELECT
        ciudad_envio,
        metodo_pago,
        total_uso,
        MAX(total_uso) OVER (
            PARTITION BY ciudad_envio
        ) AS max_usos
    FROM conteo_pagos
)
SELECT
    ciudad_envio,
    MIN(metodo_pago) AS metodo_pago,
    MAX(total_uso) AS total_usos
FROM maximos_ciudad
WHERE total_uso = max_usos
GROUP BY ciudad_envio
ORDER BY
    total_usos DESC, ciudad_envio ASC;


ciudad_envio,metodo_pago,total_usos
Trujillo,Yape,14
Cajamarca,Tarjeta,13
Piura,Tarjeta,13
Lima,Tarjeta,12
Tacna,Tarjeta,10
Arequipa,Tarjeta,9
Cusco,Tarjeta,9
Puno,Tarjeta,9
Chiclayo,Yape,8
Huancayo,Tarjeta,7


### 30. Días promedio entre registro y primera compra por segmento

Calcula, por segmento de cliente, el promedio de días transcurridos entre la fecha de registro y la primera compra.

In [0]:
%sql
-- Escribe tu consulta aquí
WITH primera_compra AS (
    SELECT
        c.cliente_id,
        c.segmento,
        DATEDIFF(
            MIN(p.fecha_pedido),
            c.fecha_registro
        ) AS dias_hasta_compra
    FROM clientes c
    INNER JOIN pedidos p ON p.cliente_id = c.cliente_id
    WHERE p.estado != 'cancelado'
    GROUP BY
        c.cliente_id, c.segmento, c.fecha_registro
)
SELECT
    segmento,
    ROUND(AVG(dias_hasta_compra), 2) AS promedio_dias
FROM primera_compra
GROUP BY segmento
ORDER BY promedio_dias ASC;


segmento,promedio_dias
Persona,186.86
Empresa,207.67
Pyme,246.53
